# P-025: Reduced-Feature Model

**Decision question:** Can we build a cleaner model without high-missingness features and still maintain panel rankings?

P-023 showed that high-missingness features are expendable. This packet builds and validates a clean model dropping all features with >10% missingness, then compares it to the full 16-feature model on both tau-b and panel device rankings.

A reduced-feature model is valuable for partners who cannot provide all 16 measurements.

In [1]:

import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'

# Compute missingness
print("Feature missingness:")
miss_rates = {}
for f in FEATURES:
    rate = df[f].isna().mean()
    miss_rates[f] = rate
    print(f"  {f:<45} {rate:>6.1%}")

# Split features by missingness threshold
THRESHOLD = 0.10
full_features = FEATURES
clean_features = [f for f in FEATURES if miss_rates[f] <= THRESHOLD]
dropped = [f for f in FEATURES if miss_rates[f] > THRESHOLD]

print(f"\nFull model: {len(full_features)} features")
print(f"Clean model (<=10% missing): {len(clean_features)} features")
print(f"Dropped: {dropped}")

ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)
PANEL = [850, 1320, 119]


Feature missingness:
  Perovskite_band_gap                            30.7%
  Pb                                              0.2%
  Sn                                              0.2%
  I                                               0.1%
  Br                                              0.1%
  Cl                                              0.1%
  MA                                              0.1%
  FA                                              0.1%
  Cs                                              0.1%
  first_Prvskt_annealing_temperature              7.6%
  first_Prvskt_thermal_annealing_time             9.0%
  Perovskite_thickness                           65.2%
  Cell_area_measured                              2.7%
  JV_default_Voc                                  3.7%
  JV_default_Jsc                                  3.3%
  JV_default_FF                                   3.8%

Full model: 16 features
Clean model (<=10% missing): 14 features
Dropped: ['Perovskite_band_gap', 

In [2]:

# Compare full vs reduced across 20 splits
N_SPLITS = 20
y = np.log1p(df[TARGET])

results = {'full': {'taus': [], 'panel': defaultdict(lambda: {'app': 0, 'top20': 0})},
           'clean': {'taus': [], 'panel': defaultdict(lambda: {'app': 0, 'top20': 0})}}

for seed in range(N_SPLITS):
    for label, feats in [('full', full_features), ('clean', clean_features)]:
        X = df[feats].fillna(0)
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
        
        model = ExtraTreesRegressor(**ET_PARAMS)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        
        tau, _ = kendalltau(y_te, preds)
        results[label]['taus'].append(tau)
        
        # Panel top-20
        test_idx = X_te.index.tolist()
        ranked = sorted(zip(test_idx, preds), key=lambda x: -x[1])
        top20 = set([idx for idx, _ in ranked[:20]])
        
        for d in PANEL:
            if d in test_idx:
                results[label]['panel'][d]['app'] += 1
                if d in top20:
                    results[label]['panel'][d]['top20'] += 1

    if (seed + 1) % 10 == 0:
        print(f"  Completed {seed+1}/{N_SPLITS} splits")

# Print comparison
print("\n" + "=" * 70)
print("FULL vs REDUCED MODEL COMPARISON")
print("=" * 70)
print(f"{'Model':<15} {'Features':>8} {'Mean tau-b':>11} {'Std':>8} {'Dev 850':>10} {'Dev 1320':>10} {'Dev 119':>10}")
print("-" * 70)

for label, feats in [('full', full_features), ('clean', clean_features)]:
    r = results[label]
    mean_tau = np.mean(r['taus'])
    std_tau = np.std(r['taus'])
    rates = {}
    for d in PANEL:
        p = r['panel'][d]
        rates[d] = f"{p['top20']}/{p['app']}" if p['app'] > 0 else "N/A"
    print(f"{label:<15} {len(feats):>8} {mean_tau:>11.4f} {std_tau:>8.4f} "
          f"{rates[850]:>10} {rates[1320]:>10} {rates[119]:>10}")

delta = np.mean(results['full']['taus']) - np.mean(results['clean']['taus'])
print(f"\nDelta (full - clean): {delta:+.4f} tau-b")


  Completed 10/20 splits


  Completed 20/20 splits

FULL vs REDUCED MODEL COMPARISON
Model           Features  Mean tau-b      Std    Dev 850   Dev 1320    Dev 119
----------------------------------------------------------------------
full                  16      0.2999   0.0420        4/4        3/3        4/4
clean                 14      0.2875   0.0396        4/4        3/3        4/4

Delta (full - clean): +0.0124 tau-b


In [3]:

# Save and status
save_rows = []
for label, feats in [('full', full_features), ('clean', clean_features)]:
    r = results[label]
    row = {'model': label, 'n_features': len(feats), 
           'mean_tau': np.mean(r['taus']), 'std_tau': np.std(r['taus'])}
    for d in PANEL:
        p = r['panel'][d]
        row[f'panel_{d}_rate'] = p['top20'] / p['app'] if p['app'] > 0 else float('nan')
    save_rows.append(row)
pd.DataFrame(save_rows).to_csv('Packet_P025_Reduced_Feature_Model.csv', index=False)
print("Saved: Packet_P025_Reduced_Feature_Model.csv")

# Status
abs_delta = abs(delta)
# Check panel survival for clean model
panel_ok = True
for d in PANEL:
    p = results['clean']['panel'][d]
    if p['app'] > 0 and p['top20'] / p['app'] < 0.80:
        panel_ok = False

if abs_delta <= 0.03 and panel_ok:
    status = "Confirmed"
elif abs_delta <= 0.05:
    status = "Promising"
else:
    status = "Negative"

print(f"\n{'=' * 60}")
print(f"P-025 STATUS: {status}")
print(f"{'=' * 60}")
print(f"Tau-b delta: {delta:+.4f} (full - clean)")
print(f"Panel survival in clean model: {'YES' if panel_ok else 'NO'}")
print(f"Dropped features: {dropped}")
print(f"Clean model retains {len(clean_features)} features with <=10% missingness")
if status == "Confirmed":
    print("The reduced model is viable for partners who can't provide all measurements.")


Saved: Packet_P025_Reduced_Feature_Model.csv

P-025 STATUS: Confirmed
Tau-b delta: +0.0124 (full - clean)
Panel survival in clean model: YES
Dropped features: ['Perovskite_band_gap', 'Perovskite_thickness']
Clean model retains 14 features with <=10% missingness
The reduced model is viable for partners who can't provide all measurements.
